# Cross-Domain QLoRA Fine-Tuning: Qwen2.5-0.5B

Fine-tune on AG News / GoEmotions / LedGAR using standard HuggingFace QLoRA.

**Change `DOMAIN` in cell 2 to switch domains.**

In [ ]:
!pip install -q transformers datasets peft bitsandbytes accelerate trl scikit-learn

In [ ]:
# =============================================
# CONFIGURATION — change DOMAIN here
# =============================================
DOMAIN = "ag_news"  # Options: "ag_news", "go_emotions", "ledgar"
SEED = 42
TRAIN_SIZE = 5000
TEST_SIZE = 1000
MAX_SEQ_LEN = 512
NUM_EPOCHS = 3
LORA_RANK = 64
LORA_ALPHA = 128
LR = 2e-4
BATCH_SIZE = 4
GRAD_ACCUM = 8
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
# =============================================

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Domain: {DOMAIN} | Seed: {SEED}")

In [ ]:
from datasets import load_dataset
from collections import Counter
import math, random
random.seed(SEED)

if DOMAIN == "ag_news":
    ds = load_dataset("ag_news")
    label_names = ["World", "Sports", "Business", "Sci/Tech"]
    sys_prompt = "Classify this news article into one category. Reply with ONLY the category name."
    def get_text_label(ex): return ex["text"], label_names[ex["label"]]
    train_split, test_split = "train", "test"

elif DOMAIN == "go_emotions":
    ds = load_dataset("google-research-datasets/go_emotions", "simplified")
    label_names = ["admiration","amusement","anger","annoyance","approval","caring",
        "confusion","curiosity","desire","disappointment","disapproval","disgust",
        "embarrassment","excitement","fear","gratitude","grief","joy","love",
        "nervousness","neutral","optimism","pride","realization","relief",
        "remorse","sadness","surprise"]
    sys_prompt = "Identify the primary emotion in this text. Reply with ONLY the emotion name."
    def get_text_label(ex): return ex["text"], label_names[ex["labels"][0]]
    train_split, test_split = "train", "test"

elif DOMAIN == "ledgar":
    ds = load_dataset("lex_glue", "ledgar")
    label_names = ds["train"].features["label"].names
    sys_prompt = "Classify this legal provision. Reply with ONLY the provision type."
    def get_text_label(ex): return ex["text"], label_names[ex["label"]]
    train_split, test_split = "train", "validation"

# Prepare data
raw_train = ds[train_split].shuffle(seed=SEED).select(range(min(TRAIN_SIZE, len(ds[train_split]))))
raw_test = ds[test_split].shuffle(seed=SEED).select(range(min(TEST_SIZE, len(ds[test_split]))))

# Compute entropy
train_labels = [get_text_label(ex)[1] for ex in raw_train]
lc = Counter(train_labels)
total = sum(lc.values())
entropy = -sum((c/total)*math.log2(c/total) for c in lc.values())

print(f"Domain: {DOMAIN} | Classes: {len(label_names)} | H(Y): {entropy:.2f} bits")
print(f"Train: {len(raw_train)} | Test: {len(raw_test)}")
print(f"Top 5: {lc.most_common(5)}")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_dropout=0.0,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Format training data
def format_example(example):
    text, label = get_text_label(example)
    messages = [
        {"role": "system", "content": sys_prompt},
        {"role": "user", "content": text[:400]},
        {"role": "assistant", "content": label}
    ]
    formatted = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": formatted}

train_formatted = raw_train.map(format_example)
print(f"Sample:\n{train_formatted[0]['text'][:300]}...")

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
import time

args = TrainingArguments(
    output_dir=f"/kaggle/working/{DOMAIN}_model",
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    fp16=True,
    logging_steps=25,
    save_strategy="no",
    seed=SEED,
    report_to="none",
    gradient_checkpointing=True,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_formatted,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    packing=False,
    args=args,
)

t0 = time.time()
trainer.train()
train_min = (time.time() - t0) / 60
print(f"\n✅ Training done in {train_min:.1f} minutes")

In [ ]:
# Evaluate
from sklearn.metrics import f1_score
from tqdm import tqdm
import json

model.eval()
preds_list, labels_list, raw_out = [], [], []

for i, example in enumerate(tqdm(raw_test, desc="Eval")):
    text, label = get_text_label(example)
    messages = [
        {"role": "system", "content": sys_prompt},
        {"role": "user", "content": text[:400]}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LEN).to("cuda")

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=32, do_sample=False, temperature=0.01)
    
    resp = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    preds_list.append(resp)
    labels_list.append(label)
    raw_out.append({"label": label, "predict": resp})

f1 = f1_score(labels_list, preds_list, average="macro", zero_division=0)
acc = sum(p == l for p, l in zip(preds_list, labels_list)) / len(labels_list)

print(f"\n{'='*50}")
print(f"Domain: {DOMAIN} | H(Y): {entropy:.2f}")
print(f"Strict Macro F1: {f1:.4f}")
print(f"Accuracy: {acc:.4f}")
print(f"Train: {train_min:.1f} min")
print(f"{'='*50}")

# Save
results = {"domain": DOMAIN, "seed": SEED, "entropy": round(entropy,2),
           "strict_macro_f1": round(f1,4), "accuracy": round(acc,4),
           "train_samples": len(raw_train), "test_samples": len(raw_test),
           "num_classes": len(label_names), "train_time_min": round(train_min,1),
           "model": MODEL_NAME, "lora_rank": LORA_RANK}
with open(f"/kaggle/working/{DOMAIN}_results.json", "w") as f:
    json.dump(results, f, indent=2)
with open(f"/kaggle/working/{DOMAIN}_predictions.jsonl", "w") as f:
    for item in raw_out: f.write(json.dumps(item)+"\n")
print(f"Saved to /kaggle/working/{DOMAIN}_results.json")

In [ ]:
# Quick look at predictions
for item in raw_out[:15]:
    s = "✅" if item["label"] == item["predict"] else "❌"
    print(f"{s} Label: {item['label']:20s} | Pred: {item['predict'][:30]}")